In [ ]:
import requests
import pandas as pd
import re
import unicodedata
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from difflib import SequenceMatcher

HEADERS = {"User-Agent": "Mozilla/5.0"}

def normalize(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    s = s.replace("&", " and ")
    s = re.sub(r"['’]", "", s)
    s = re.sub(r"[-_/]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, normalize(a), normalize(b)).ratio()

def build_beautetest_brand_index():
    url = "https://www.beaute-test.com/marques"
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")
    rows = []

    for a in soup.select('a[href^="/marque/"]'):
        href = a.get("href", "").strip()
        name = a.get_text(strip=True)
        if not name:
            continue

        rows.append({
            "brand_name": name,
            "brand_name_norm": normalize(name),
            "brand_path": href
        })

    df = pd.DataFrame(rows).drop_duplicates(subset=["brand_path"]).reset_index(drop=True)
    return df

def resolve_keyword_to_brand(keyword: str, brand_index: pd.DataFrame):
    key_norm = normalize(keyword)

    # exact normalized match first
    exact = brand_index[brand_index["brand_name_norm"] == key_norm]
    if not exact.empty:
        return exact.iloc[0].to_dict()

    # fallback: best fuzzy match
    scored = brand_index.copy()
    scored["score"] = scored["brand_name"].apply(lambda x: similarity(keyword, x))
    best = scored.sort_values("score", ascending=False).iloc[0]

    return {
        "brand_name": best["brand_name"],
        "brand_name_norm": best["brand_name_norm"],
        "brand_path": best["brand_path"],
        "score": round(best["score"], 3)
    }

In [ ]:
KEYWORDS_BEAUTETEST_FR = {
    "Luxe":      ["dior", "guerlain", "givenchy", "yves-saint-laurent"],
    "Prestige":  ["lancome", "clarins", "giorgio-armani"],
    "Masstige":  ["la-roche-posay", "vichy", "cerave"],
    "Mass":      ["loreal-paris", "garnier", "gemey-maybelline"],  # ← swap
}

In [ ]:
brand_index = build_beautetest_brand_index()

resolved = []
for tier, keywords in KEYWORDS_BEAUTETEST_FR.items():
    for kw in keywords:
        hit = resolve_keyword_to_brand(kw, brand_index)
        resolved.append({
            "tier": tier,
            "keyword": kw,
            "resolved_brand": hit["brand_name"],
            "brand_path": hit["brand_path"],
            "score": hit.get("score", 1.0)
        })

resolved_df = pd.DataFrame(resolved)
resolved_df